## 使用者輸入查詢主題

In [ ]:
research_topic = input("請輸入您想研究的主題： ")

## Gemini API Key (事先申請)

### 從 `.env` 載入

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
print(f"API Key Found: {os.environ.get('GEMINI_API_KEY') is not None}")

### 用原生Gemini確認是否有通

In [ ]:
'''
from google import genai
client = genai.Client()
response = client.models.generate_content(model="gemini-2.5-flash", contents="Hello")
print(response.text)
'''

## LangChain + Gemini

### Gemini Setting

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI # 引入 Gemini 聊天模型類別 也可以使用其他LLM 可參考：https://docs.langchain.com/oss/python/integrations/providers/overview
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 選擇模型 (Model)
# 使用 Google 的 Chat 模型，這裡我們選擇快速且高性能的 gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=os.getenv("GEMINI_API_KEY"))

# 2. 建立提示模板 (Prompt Template)
prompt_text = "你是一位專業的學術文獻總結專家，請用繁體中文，針對使用者提供的內容，進行精簡且重點突出的摘要。"
# prompt_text = "你是一位專業的學術文獻總結專家，請根據使用者提問時使用的語言，針對提供的內容，進行精簡且重點突出的摘要。"
prompt = ChatPromptTemplate.from_messages([
    ("system", prompt_text),
    ("user", "{text_content}") # {text_content} 是一個變數
])

# 3. 建立輸出解析器 (Output Parser)
output_parser = StrOutputParser()

# 4. 串聯組件 (Chain)
# 整個流程：輸入 -> 提示模板 -> Gemini 模型 -> 輸出解析器
simple_chain = prompt | llm | output_parser

# 5. 執行 Chain
article_text = "Transformer 架構自 2017 年問世以來，已成為自然語言處理和電腦視覺領域的基石。最新的研究傾向於減少注意力機制的計算複雜度，並將其應用擴展到更長的序列任務上。此外，多模態 Transformer 模型的發展，如能處理文本和圖像的模型，正在成為新的研究焦點。"

print("--- 正在呼叫 Gemini 模型進行摘要 ---")
result = simple_chain.invoke({"text_content": article_text})

print("\n--- Gemini 摘要結果 ---")
print(result)

### Search from ArXiv

In [ ]:
import arxiv
from langchain_core.documents import Document
from typing import List

# 💡 請修改您的研究主題和參數 💡
# research_topic = "Retrieval-Augmented Generation"  
max_results = 5  

print(f"--- 正在使用純粹 arxiv 套件搜尋主題 '{research_topic}' 的最新 {max_results} 篇論文 ---")

def load_arxiv_documents(query: str, max_results: int) -> List[Document]:
    """使用 arxiv 套件查詢並轉換為 LangChain Document"""
    client = arxiv.Client()
    
    # 建立查詢條件
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate # 依提交日期排序
    )
    
    # 執行查詢
    results = client.results(search)
    
    docs = []
    for result in results:
        # 手動將每個結果轉換為 LangChain Document
        doc = Document(
            # 摘要作為主要內容 (page_content)
            page_content=result.summary,
            # 將其他元數據放入 metadata 字典
            metadata={
                "Title": result.title,
                "Authors": ", ".join([a.name for a in result.authors]),
                "Published": result.published.strftime("%Y-%m-%d"),
                "URL": result.entry_id,
                "PDF_URL": result.pdf_url,
                "Categories": ", ".join(result.categories),
            }
        )
        docs.append(doc)
    return docs

# 執行載入
docs = load_arxiv_documents(research_topic, max_results)

print(f"✅ 成功載入 {len(docs)} 篇論文。")
print(f"論文標題:")
for i in range(len(docs)):
    print(f"{i+1}. {docs[i].metadata['Title']} ({docs[i].metadata['Published']})")
print("-" * 30)

# 顯示第一篇文檔的內容和元數據結構
print("--- Document Content (Abstract) ---")
print(docs[0].page_content[:500])
print("\n--- Document Metadata ---")
print(docs[0].metadata)

### (X) ArXiv Loader

In [ ]:
'''
from langchain_community.document_loaders import ArxivLoader
from langchain_community.utilities import ArxivAPIWrapper

# 💡 請修改您的研究主題和參數 💡
research_topic = "Retrieval-Augmented Generation"  # 例如：RAG 技術
max_results = 5  # 限制只搜尋最新的 5 篇論文

# print(f"--- 正在從 arXiv 搜尋並載入主題 '{research_topic}' 的最新 {max_results} 篇論文 ---")
arxiv_api_wrapper = ArxivAPIWrapper(
    load_max_docs=max_results, 
    # 關鍵設定：設定為 False 避免嘗試下載 PDF
    doc_content_chars_max=0,  
    # 另一個參數 (如果需要)：設定為 False 避免下載整個 PDF (但上面的參數更直接)
    keep_pdf_on_disk=False 
)

loader = ArxivLoader(
    query=research_topic,  
    load_max_docs=max_results,
    # 💡 將自定義的 API wrapper 傳入 loader 中
    arxiv_api_wrapper=arxiv_api_wrapper
)

docs = loader.load()


print(f"✅ 成功載入 {len(docs)} 篇論文。")
print(f"首篇論文標題: {docs[0].metadata['Title']}")
print("-" * 30)

# 顯示第一篇文檔的內容和元數據結構
print(docs[0].page_content[:500]) # 這是摘要/前言部分
print(docs[0].metadata)
'''

### LangChain Text Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_document(docs):
    # 設置文本切分器
    # chunk_size: 每個文本塊的大小（例如 1000 個字符）
    # chunk_overlap: 文本塊之間的重疊量，有助於保持上下文的連續性
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, 
        chunk_overlap=150, 
        add_start_index=True # 添加開始索引，方便追溯原始文件位置
    )

    # 將文件列表 (docs) 切分成更小的文本塊
    all_splits = text_splitter.split_documents(docs)

    # print(f"原始文檔數量: {len(docs)}")
    # print(f"切分後的文本塊總數: {len(all_splits)}")
    # print(f"範例文本塊 (Chunk) 長度: {len(all_splits[0].page_content)} 個字符")
    return all_splits

### Gemini Embedding

In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# --- 確保這段程式碼已經運行 ---
# 1. 初始化嵌入模型 (Embedding Model)
# 確保 os.environ.get("GEMINI_API_KEY") 能夠取到金鑰
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=os.getenv("GEMINI_API_KEY"))

def  create_vectorstore(all_splits):
    # print("\n--- 正在創建向量資料庫 ---")
    # 2. 創建 FAISS 向量儲存
    # 這一步會將 all_splits 向量化，並將結果存入 vectorstore 變數中
    vectorstore = FAISS.from_documents(
        documents=all_splits,
        embedding=embedding_model
    )

    # print("✅ 向量資料庫建立成功！知識庫已就緒。")
    # -------------------------------
    # 根據您先前建立的 FAISS 向量儲存，創建檢索器
    # k=3 表示每次查詢時，檢索器會返回最相關的 3 個文本塊
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # print("✅ 檢索器創建成功。")
    return retriever

### Agent模板

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """你是一位專業的研究助理，請根據以下提供的『上下文資料』，詳細且準確地回答用戶的問題。
請僅使用上下文中的資訊來回答，如果資訊不足，請表明「資料不足，無法回答」。
請使用繁體中文。

--- 上下文資料 ---
{context}

--- 使用者問題 ---
{question}
"""

AGENT_PROMPT = ChatPromptTemplate.from_template(template)

### RAG

In [ ]:
# 定義一個格式化函數，用於將檢索到的文檔（Documents）轉換成單一的字串（Context）
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def rag_chain_setup(retriever):
    # 串聯完整的 RAG Chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | AGENT_PROMPT
        | llm
        | StrOutputParser()
    )

    # print("✅ 完整的 RAG 檢索與生成鏈（RAG Chain）建構完成。")
    return rag_chain

In [ ]:
def response_the_ques(docs, question):
    all_splits = split_document(docs)
    retriever = create_vectorstore(all_splits)
    rag_chain = rag_chain_setup(retriever)
    print(f"\n--- 正在向知識庫查詢問題：{question} ---")
    return rag_chain.invoke(question)

### 提問

In [ ]:
question = f"請先幫我簡單摘要 {research_topic} 是什麼？"

# 運行 RAG Chain
# Chain 會自動：檢索 -> 構建提示 -> 傳給 Gemini -> 獲取最終答案
response = response_the_ques(docs, question)
print(response)

In [ ]:
print(f"\n--- 正在整理各篇重點 ---")

for i in range(1, max_results + 1):
    study = []
    study.append(docs[i-1])
    print(f"\n--- 論文 {i} 重點 ---")
    print(f"標題: {docs[i-1].metadata['Title']}")
    print(f"作者: {docs[i-1].metadata['Authors']}")
    print(f"發表日期: {docs[i-1].metadata['Published']}")
    print(f"分類: {docs[i-1].metadata['Categories']}")
    print(f"URL: {docs[i-1].metadata['URL']}")
    question = f"請幫我把搜尋到的本篇論文，把重點條列整理出來。"
    response = response_the_ques(study, question)
    print(response)

In [ ]:
question = f" 請問 {research_topic} 有哪些主要的優化方向？根據最新的論文，哪種優化方法的效果最好？"

response = response_the_ques(docs, question)
print(response)

In [ ]:
while True:
    arr = list(int(x)-1 for x in input("請輸入您想查詢哪幾篇論文（1~5篇，用空格分開）：").split())
    if any(i < 0 or i >= len(docs) for i in arr):
        print("輸入的論文編號無效，請重新輸入。")
    else:
        arr = sorted(arr)
        print(f"您選擇了以下論文：")
        for i in arr:
            print(f"{i+1}. {docs[i].metadata['Title']}")
        break

question = input("請輸入您想詢問的問題(結束請輸入exit)： ")
response = response_the_ques([docs[i] for i in arr], question)
print(response)

### 存成PDF (尚未完成)

In [ ]:
'''
from fpdf import FPDF

# 假設您的 RAG 輸出已經儲存在 response 變數中
rag_content = response # 替換為您的 RAG 輸出變數

# --- 輸出 PDF 的程式碼 ---

# 1. 建立 FPDF 物件
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15) # 設定自動分頁

# 2. 加入一頁
pdf.add_page()

# 3. 設定字體
# 注意：如果您的內容有中文，必須載入支援中文的字體 (如 NotoSansTC)。
# 這裡先使用內建的 Arial 示範，中文可能無法正常顯示。
pdf.set_font("Arial", "B", 16)
pdf.cell(0, 10, txt="RAG 深度研究報告摘要", ln=1, align='C')

pdf.set_font("Arial", size=12)

# 4. 將多行文字寫入 PDF
# multi_cell 函式可以自動換行並處理多段內容
# 參數 (w=0: 寬度為頁面寬度, h=5: 行高, txt: 內容)
pdf.multi_cell(w=0, h=5, txt=rag_content)

# 5. 輸出 PDF 檔案
pdf.output("RAG_Research_Summary.pdf")

print("PDF 檔案已成功生成為：RAG_Research_Summary.pdf")
'''